# 蛋白质功能预测 — Baseline

氨基酸组成（20 维频率 + 序列长度）+ 逐标签 RandomForest。

- 输入：`data/train.csv`（含标签）、`data/test.csv`（测试集序列）
- 输出：`submission.csv`（`protein_id, label_0..label_499`）

## 1. 导入

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

## 2. 读取数据

In [ ]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')
label_cols = [c for c in train.columns if c.startswith('label_')]
print(f'train: {len(train)}, test: {len(test)}, labels: {len(label_cols)}')
train.head()

## 3. 特征工程：氨基酸组成 + 序列长度

In [ ]:
def extract_features(sequences):
    n, n_aa = len(sequences), len(AMINO_ACIDS)
    feats = np.zeros((n, n_aa + 1), dtype=np.float32)
    idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
    for i, seq in enumerate(sequences):
        L = len(seq)
        feats[i, -1] = np.log1p(L)
        if L == 0:
            continue
        for c in seq:
            if c in idx:
                feats[i, idx[c]] += 1
        feats[i, :n_aa] /= L
    return feats

In [ ]:
X_train = extract_features(train['sequence'])
X_test  = extract_features(test['sequence'])
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

## 4. 逐标签训练 RandomForest

In [ ]:
n_estimators, max_depth, min_pos = 50, 12, 10
y_pred = np.zeros((len(test), len(label_cols)), dtype=np.int8)
t0 = time.time()
for i, col in enumerate(label_cols):
    y = train[col].values.astype(int)
    if y.sum() < min_pos:
        continue
    rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                n_jobs=-1, random_state=42, class_weight='balanced')
    rf.fit(X_train, y)
    y_pred[:, i] = rf.predict(X_test)
    if (i + 1) % 50 == 0:
        el = time.time() - t0
        print(f'[{i+1}/{len(label_cols)}] elapsed={el:.0f}s, ETA={el/(i+1)*(len(label_cols)-i-1):.0f}s')
print(f'done in {time.time()-t0:.0f}s; positive predictions: {y_pred.mean():.2%}')

## 5. 保存提交

In [ ]:
submission = test[['protein_id']].copy()
for i, col in enumerate(label_cols):
    submission[col] = y_pred[:, i]
submission.to_csv('submission.csv', index=False)
print(f'saved submission.csv: {len(submission)} rows')
submission.head()